# 🚀 DBT Tests --- Types, Usage & Production Examples

------------------------------------------------------------------------

# 🧠 What are DBT Tests?

DBT tests are **data quality checks** written in SQL that ensure your
data is:

-   ✅ Correct\
-   ✅ Consistent\
-   ✅ Reliable

👉 If a test returns **rows → FAIL ❌**\
👉 If it returns **0 rows → PASS ✅**

------------------------------------------------------------------------

# ⚙️ How DBT Tests Work Internally

DBT converts tests into SQL like:

``` sql
SELECT *
FROM table
WHERE condition_is_invalid
```

👉 If any rows returned → data issue

------------------------------------------------------------------------

# 🧩 Types of DBT Tests

------------------------------------------------------------------------

# 🧪 1. Generic Tests (Built-in Tests)

👉 Defined in `schema.yml` / `properties.yml`

------------------------------------------------------------------------

## 🔹 1.1 unique

### 📌 Ensures:

-   No duplicate values

``` yaml
models:
  - name: stg_orders
    columns:
      - name: id
        tests:
          - unique
```

------------------------------------------------------------------------

## 🔹 1.2 not_null

### 📌 Ensures:

-   No NULL values

``` yaml
models:
  - name: stg_orders
    columns:
      - name: id
        tests:
          - not_null
```

------------------------------------------------------------------------

## 🔹 1.3 accepted_values

### 📌 Ensures:

-   Values belong to allowed list

``` yaml
models:
  - name: orders
    columns:
      - name: status
        tests:
          - accepted_values:
              values: ['placed', 'shipped', 'delivered']
```

------------------------------------------------------------------------

## 🔹 1.4 relationships

### 📌 Ensures:

-   Foreign key integrity

``` yaml
models:
  - name: orders
    columns:
      - name: customer_id
        tests:
          - relationships:
              to: ref('customers')
              field: id
```

------------------------------------------------------------------------

# 🧠 Summary of Generic Tests

  Test Type         Purpose
  ----------------- -----------------------
  unique            No duplicates
  not_null          No null values
  accepted_values   Valid category values
  relationships     Foreign key integrity

------------------------------------------------------------------------

# 🧪 2. Singular Tests (Custom SQL Tests)

👉 Written manually in `tests/` folder

------------------------------------------------------------------------

## 📌 Example

``` sql
-- tests/test_positive_amount.sql

SELECT *
FROM {{ ref('orders') }}
WHERE amount < 0
```

👉 Fails if: - Any negative amount exists

------------------------------------------------------------------------

# 🧠 Key Idea

-   Generic tests → reusable\
-   Singular tests → custom logic

------------------------------------------------------------------------

# 🧪 3. Custom Generic Tests (Advanced)

👉 You can create reusable test macros

------------------------------------------------------------------------

## Step 1: Create Macro

``` sql
-- macros/test_positive.sql

{% test positive(model, column_name) %}

SELECT *
FROM {{ model }}
WHERE {{ column_name }} < 0

{% endtest %}
```

------------------------------------------------------------------------

## Step 2: Use in YAML

``` yaml
models:
  - name: orders
    columns:
      - name: amount
        tests:
          - positive
```

------------------------------------------------------------------------

# ⚙️ How to Run Tests

------------------------------------------------------------------------

## Run All Tests

``` bash
dbt test
```

------------------------------------------------------------------------

## Run Specific Model Tests

``` bash
dbt test -s stg_orders
```

------------------------------------------------------------------------

## Run Specific Test Type

``` bash
dbt test -s test_type:generic
```

------------------------------------------------------------------------

# 🏗️ Production-Level Example

------------------------------------------------------------------------

## 📁 Model: stg_orders.sql

``` sql
SELECT
    id,
    customer_id,
    amount,
    status
FROM raw.orders
```

------------------------------------------------------------------------

## 📁 properties.yml

``` yaml
models:
  - name: stg_orders
    columns:
      - name: id
        tests:
          - unique
          - not_null

      - name: status
        tests:
          - accepted_values:
              values: ['placed', 'shipped', 'delivered']

      - name: customer_id
        tests:
          - relationships:
              to: ref('customers')
              field: id
```

------------------------------------------------------------------------

## 📁 Custom Test

``` sql
-- tests/test_amount_positive.sql

SELECT *
FROM {{ ref('stg_orders') }}
WHERE amount < 0
```

------------------------------------------------------------------------

# 🧠 Test Execution Flow

1.  DBT reads tests from YAML + SQL\
2.  Converts to SQL queries\
3.  Executes in warehouse\
4.  Returns results

------------------------------------------------------------------------

# 🔥 Important Concepts

------------------------------------------------------------------------

## 1. Test Failure

👉 If rows returned:

Test FAILED ❌

------------------------------------------------------------------------

## 2. Severity (Advanced)

``` yaml
tests:
  - not_null:
      severity: warn
```

👉 Options: - error (default) - warn

------------------------------------------------------------------------

## 3. Store Failures

``` yaml
tests:
  - unique:
      store_failures: true
```

👉 Saves failed rows in table

------------------------------------------------------------------------

# 🚀 Best Practices (Production)

-   Always test primary keys (unique + not_null)\
-   Validate dimensions using accepted_values\
-   Use relationships for joins\
-   Add custom tests for business rules\
-   Use severity wisely

------------------------------------------------------------------------

# 🎯 Interview Points

-   DBT tests are SQL queries\
-   2 types: Generic & Singular\
-   Generic tests defined in YAML\
-   Singular tests written in SQL\
-   Fail if rows returned

------------------------------------------------------------------------

# ⚡ Final Summary

DBT Tests: - Ensure data quality\
- Built-in + custom\
- Executed inside warehouse\
- Critical for production pipelines


# 🚀 DBT Testing --- Advanced (dbt-utils + Production Strategy + Debugging)

------------------------------------------------------------------------

# 🧠 1. dbt-utils Package (Very Important 🔥)

## 📌 What is dbt-utils?

-   Open-source package with reusable macros & tests
-   Widely used in real projects

------------------------------------------------------------------------

## ⚙️ Installation

Add in `packages.yml`:

``` yaml
packages:
  - package: dbt-labs/dbt_utils
    version: 1.1.1
```

Run:

``` bash
dbt deps
```

------------------------------------------------------------------------

## 🔥 Common dbt-utils Tests

------------------------------------------------------------------------

### 1. unique_combination_of_columns

``` yaml
models:
  - name: orders
    tests:
      - dbt_utils.unique_combination_of_columns:
          combination_of_columns:
            - order_id
            - customer_id
```

👉 Ensures composite uniqueness

------------------------------------------------------------------------

### 2. not_null_proportion

``` yaml
- dbt_utils.not_null_proportion:
    column_name: email
    at_least: 0.95
```

👉 Ensures 95% values are not null

------------------------------------------------------------------------

### 3. accepted_range

``` yaml
- dbt_utils.accepted_range:
    column_name: amount
    min_value: 0
    max_value: 100000
```

👉 Validates numeric range

------------------------------------------------------------------------

### 4. relationships_where

``` yaml
- dbt_utils.relationships_where:
    to: ref('customers')
    field: id
    condition: "is_active = true"
```

👉 Conditional FK validation

------------------------------------------------------------------------

# 🏗️ 2. Real Production Test Strategy

------------------------------------------------------------------------

## 🧩 Layer-wise Testing

### 🧹 Staging Layer

-   unique
-   not_null

------------------------------------------------------------------------

### 🔄 Intermediate Layer

-   relationships
-   basic aggregations

------------------------------------------------------------------------

### 📊 Mart Layer

-   business logic tests
-   custom tests

------------------------------------------------------------------------

## 🔥 What Companies Actually Test

-   Primary keys (100%)
-   Foreign keys (joins)
-   Business rules
-   Null thresholds
-   Data freshness

------------------------------------------------------------------------

## 📌 Example Strategy

``` yaml
models:
  - name: mart_orders
    columns:
      - name: order_id
        tests: [unique, not_null]

      - name: customer_id
        tests:
          - relationships:
              to: ref('customers')
              field: id

      - name: amount
        tests:
          - dbt_utils.accepted_range:
              min_value: 0
```

------------------------------------------------------------------------

# 🛠️ 3. Debugging Failing Tests (Step-by-Step)

------------------------------------------------------------------------

## Step 1: Run Tests

``` bash
dbt test
```

------------------------------------------------------------------------

## Step 2: Identify Failure

Example:

``` text
FAIL 1 rows returned
```

------------------------------------------------------------------------

## Step 3: Locate Query

Check:

    target/compiled/

👉 Find compiled test SQL

------------------------------------------------------------------------

## Step 4: Run Query in Warehouse

``` sql
SELECT * FROM table WHERE issue_condition
```

------------------------------------------------------------------------

## Step 5: Analyze Root Cause

-   Null values?
-   Duplicates?
-   Bad joins?

------------------------------------------------------------------------

## Step 6: Fix Data / Logic

Options: - Fix upstream model - Adjust test logic - Add filters

------------------------------------------------------------------------

## Step 7: Re-run

``` bash
dbt test
```

------------------------------------------------------------------------

# ⚡ Advanced Debugging Tips

-   Use `store_failures: true`
-   Check logs in `logs/dbt.log`
-   Run specific test:

``` bash
dbt test -s model_name
```

------------------------------------------------------------------------

# 🧠 4. Best Practices (Real World)

-   Test critical columns only (avoid noise)
-   Use dbt-utils extensively
-   Keep tests close to models
-   Use severity wisely (warn vs error)
-   Automate in CI/CD

------------------------------------------------------------------------

# 🎯 5. Interview Cheat Points

-   dbt-utils = industry standard
-   Tests are SQL queries
-   Fail if rows returned
-   Layer-based testing strategy
-   Debug via compiled SQL

------------------------------------------------------------------------

# ⚡ Final Summary

Advanced DBT Testing: - Built-in + dbt-utils + custom - Layered testing
approach - Strong debugging workflow - Critical for production
reliability
